# 01 · Dataset + classical ML baselines (Random Forest, XGBoost)

**Goal:** build the agent-routing dataset, look at it, and train the two classical baselines.

Each ticket has free text (`query`) plus tabular metadata (`account_tier`, `channel`, `urgency_score`,
`past_failures`). The model must output a **typed decision**:

| field | type | values |
|---|---|---|
| `route_target` | choice | billing, orders, account, tech_support, fraud, general |
| `priority_level` | ordinal | low < medium < high < critical |
| `requires_human` | yes/no | True / False |

Runs fine on a CPU runtime; no GPU needed for this notebook.

## Setup

Works both on **Colab** and locally. On Colab this clones the repo and moves into it;
locally (opened from `notebooks/`) it just moves up to the repo root so `import src...` works.

In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/AdityaThakare72/typed-decision-models.git"
BRANCH = "main"

if not os.path.exists("src/dataset.py"):
    if os.path.exists("../src/dataset.py"):      # running locally from notebooks/
        os.chdir("..")
    else:                                         # fresh Colab runtime
        if not os.path.exists("typed-decision-models"):
            subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL], check=True)
        os.chdir("typed-decision-models")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

working dir: /content/typed-decision-models


Install the light dependencies (Colab already has most of them, so this is quick).

In [2]:
%pip install -q scikit-learn xgboost pandas matplotlib

## 1. Build the dataset

Two options:
- `USE_HF = False` → the **offline synthetic** builder: templated queries, deterministic (same seed → same data).
- `USE_HF = True` → **real customer-support queries** from Hugging Face (Bitext), with the same metadata + labelling rules on top.
  Note: Bitext has no fraud tickets, so the `fraud` route will be empty.

Labels come from simple rules (see `decide_priority` / `decide_requires_human` in `src/dataset.py`) plus 5% random label
noise, so no model can be perfect. The splits are saved to `data/` so every notebook uses the same rows.

In [3]:
from src.dataset import (build_synthetic_dataset, load_hf_dataset, train_val_test_split,
                         save_splits, LABEL_COLUMNS, FEATURE_COLUMNS)

USE_HF = False
N_ROWS = 3000
SEED = 42

df = load_hf_dataset(N_ROWS, SEED) if USE_HF else build_synthetic_dataset(N_ROWS, SEED)
train_df, val_df, test_df = train_val_test_split(df, seed=SEED)
save_splits(train_df, val_df, test_df, "data")
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
df.head(10)

train=2099  val=451  test=450


,query,account_tier,channel,urgency_score,past_failures,route_target,priority_level,requires_human
0,how do I cancel my annual subscription and sto...,pro,email,0.654,1,billing,medium,False
1,tracking for order 40920 has not updated in da...,pro,chat,0.347,1,orders,medium,False
2,need a copy of the invoice for order 5531 no rush,pro,chat,0.221,1,billing,medium,False
3,what are your support hours when you get a chance,free,chat,0.398,1,general,low,False
4,how long does shipping take to Tokyo no rush,free,chat,0.323,2,orders,medium,False
5,I can't log in after changing my phone number ...,pro,chat,0.373,1,account,medium,False
6,why did my card get declined when paying for a...,free,email,0.723,0,billing,medium,False
7,just wanted to say the new CSV export is great...,free,email,0.377,0,general,low,False
8,suspicious login alerts keep coming in when yo...,free,email,0.268,1,fraud,medium,True
9,need a copy of the invoice for order 1042 this...,pro,chat,0.820,1,billing,high,True


## 2. Quick EDA

First, how balanced are the three labels? Routes are roughly even by design. Priority is skewed
(few `critical` tickets) — that is why we report **macro-F1**, which weighs every class equally.

In [4]:
for column in LABEL_COLUMNS:
    print(df[column].value_counts(normalize=True).round(3).to_dict())

{'tech_support': 0.175, 'general': 0.172, 'billing': 0.169, 'orders': 0.167, 'account': 0.163, 'fraud': 0.154}
{'medium': 0.469, 'low': 0.274, 'high': 0.212, 'critical': 0.045}
{False: 0.735, True: 0.265}


How do the labels depend on the metadata? Each row below shows, per account tier, the share of tickets in each
priority level. Enterprise tickets lean towards `high`/`critical`, which is exactly the kind of signal the tabular
features must carry (the text alone can't tell you the tier).

In [5]:
import pandas as pd

priority_order = ["low", "medium", "high", "critical"]
(pd.crosstab(df["account_tier"], df["priority_level"], normalize="index")[priority_order]
   .round(2))

priority_level,low,medium,high,critical
account_tier,,,,
enterprise,0.01,0.44,0.42,0.13
free,0.42,0.45,0.11,0.02
pro,0.18,0.50,0.27,0.05


Same idea for the yes/no field: share of tickets that need a human, by route and by number of past failures.

In [6]:
print(df.groupby("route_target")["requires_human"].mean().round(2).sort_values())
print()
print(df.groupby("past_failures")["requires_human"].mean().round(2))

route_target
orders          0.11
billing         0.14
general         0.16
account         0.17
tech_support    0.17
fraud           0.89
Name: requires_human, dtype: float64

past_failures
0    0.21
1    0.23
2    0.27
3    0.96
4    0.94
5    1.00
Name: requires_human, dtype: float64


Query lengths: they are short (tickets, not essays), which matters later: TF-IDF works well on short text,
and transformer latency grows with input length.

In [7]:
lengths = df["query"].str.split().str.len()
lengths.describe().round(1)

,query
count,3000.0
mean,10.6
std,2.5
min,5.0
25%,9.0
50%,11.0
75%,12.0
max,17.0


## 3. Train Random Forest and XGBoost

Both use the same feature pipeline (`build_preprocessor` in `src/models/classical_ml.py`):
- `query` → **TF-IDF** on words + word pairs
- `account_tier`, `channel` → **one-hot**
- `urgency_score`, `past_failures` → **standardised** (mean 0, std 1)

Trees predict one label at a time, so each model trains **three classifiers** (one per output field).

In [8]:
from src.models.classical_ml import ClassicalDecisionModel

rf = ClassicalDecisionModel("rf", seed=SEED).fit(train_df)
xgb = ClassicalDecisionModel("xgb", seed=SEED).fit(train_df)
print("trained:", rf.name, xgb.name)

trained: random_forest xgboost


## 4. Benchmark

`run_benchmark` (in `src/interface.py`) runs every model on the same test set and reports:
- accuracy + macro-F1 per field, plus **exact_match** (all three fields right at once, what a router really needs)
- latency **p50 / p95** over 100 single-ticket predictions (after a short warm-up)
- model size in MB (pickled pipelines)

In [9]:
from src.interface import run_benchmark

results = run_benchmark([rf, xgb], test_df, n_latency_runs=100)
results.round(3)

evaluating random_forest ...
evaluating xgboost ...


,route_acc,route_f1,priority_acc,priority_f1,requires_acc,requires_f1,exact_match,p50_ms,p95_ms,size_mb
model,,,,,,,,,,
random_forest,0.944,0.944,0.673,0.538,0.920,0.892,0.598,83.154,133.971,67.530
xgboost,0.951,0.951,0.820,0.641,0.942,0.924,0.736,21.788,24.336,5.447


Save the results so notebook 03 (or the README) can reuse them without retraining.

In [10]:
os.makedirs("artifacts", exist_ok=True)
results.to_csv("artifacts/results_classical.csv")
print(open("artifacts/results_classical.csv").read())

model,route_acc,route_f1,priority_acc,priority_f1,requires_acc,requires_f1,exact_match,p50_ms,p95_ms,size_mb
random_forest,0.9444444444444444,0.9441037652352343,0.6733333333333333,0.5377540101984113,0.92,0.891711229946524,0.5977777777777777,83.15409100001148,133.97135009999488,67.52959728240967
xgboost,0.9511111111111111,0.9507838338506339,0.82,0.6409082408565666,0.9422222222222222,0.9236372180451128,0.7355555555555555,21.788416499987306,24.336387299965168,5.446707725524902



## 5. What did XGBoost learn?

Top features for the `priority_level` classifier, ranked by **total gain**: how much each feature reduced the loss,
summed over every split that used it.

Why not the default `feature_importances_`? That is *average* gain per split, which rewards rare TF-IDF words that
happened to make one lucky split. Urgency gets split on thousands of times with small gains each, so it looks
unimportant by average and dominant by total. Expect urgency, past failures and the tier one-hots on top, which
matches the rules in `decide_priority`.

In [ ]:
from src.models.classical_ml import feature_names

clf = xgb.pipelines["priority_level"].named_steps["clf"]
names = feature_names(xgb, "priority_level")
# get_score keys look like "f12" -> feature number 12
total_gain = clf.get_booster().get_score(importance_type="total_gain")
importance = pd.Series({names[int(key[1:])]: value for key, value in total_gain.items()})
importance.sort_values(ascending=False).head(15).round(1)

## 6. Error check: where does XGBoost get priority wrong?

A confusion table for priority. Most mistakes should be between **neighbouring** levels (medium↔high);
jumping low↔critical would be a worse error for a router.

In [12]:
from src.interface import outputs_to_frame

pred = outputs_to_frame(xgb.predict_batch(test_df))
pd.crosstab(test_df["priority_level"], pred["priority_level"],
            rownames=["true"], colnames=["predicted"]).reindex(index=priority_order, columns=priority_order, fill_value=0)

predicted,low,medium,high,critical
true,,,,
low,114,11,1,0
medium,11,191,16,1
high,0,24,63,0
critical,1,6,10,1


**Takeaways to fill in after running:**
- Route accuracy: _TODO_
- Priority macro-F1 (the hard field): _TODO_
- Latency p50/p95 on CPU: _TODO_

Next: `02_laya_base_vs_finetuned.ipynb`.